Task 1:
implement architecture of docs/dataset.mmd in base/ package

in this notebook implement scenario: 
load nerel dataset (look into NEREL_MultiTask*.ipynb)
prepare BaseNerelDatasetBundle
store bundle, load bundle to check

implement two models - random prediction of classes, tags, constant prediction of the same tag, of constant cls list

collect topic and tag metrics (look into NEREL_MultiTask*.ipynb)

stop
do not go beyond of docs/dataset.mmd architecture

# Task 1 — данные и простые модели NEREL

Реализации используют интерфейсы `docs/dataset.mmd`. Работаем с исходными словами и BIO-тегами без субтокенизации. `TextTopic` — элемент исходного multihot-вектора `cls_vec`: каждая из 30 характеристик документа является отдельной бинарной целью.

Источник `danasone/nerel` загружается так же, как в исходном ноутбуке. Его `train` делим по документам на train/test = 80/20 с seed 42. Справочник тегов строится по train. Метрики считаем только на test.

In [1]:
import json
from collections.abc import Callable, Iterable
from pathlib import Path
from random import Random
from typing import Literal, Self, cast

from datasets import Dataset as HFDataset
from datasets import load_dataset
from sklearn.metrics import f1_score, precision_score, recall_score

from base import (
    BaseDatasetStorage,
    BaseDocument,
    BaseMetric,
    BaseMetricEvaluator,
    BaseModel,
    BaseNerelDataset,
    BaseNerelDatasetBundle,
    BaseNerelExample,
    BaseTextPrediction,
    BaseTextTagFabric,
    TextTag,
    TextToken,
    TextTopic,
)

SEED = 42
TEST_SIZE = 0.2
BUNDLE_PATH = Path("artifacts/nerel_reorganized")
CLS_LABELS = [
    "WORKPLACE",
    "ALTERNATIVE_NAME",
    "WORKS_AS",
    "PARTICIPANT_IN",
    "POINT_IN_TIME",
    "TAKES_PLACE_IN",
    "HEADQUARTERED_IN",
    "ORIGINS_FROM",
    "LOCATED_IN",
    "AGENT",
    "AGE_IS",
    "HAS_CAUSE",
    "PRODUCES",
    "AWARDED_WITH",
    "PART_OF",
    "IDEOLOGY_OF",
    "MEMBER_OF",
    "CONVICTED_OF",
    "INANIMATE_INVOLVED",
    "SUBEVENT_OF",
    "SUBORDINATE_OF",
    "KNOWS",
    "MEDICAL_CONDITION",
    "PARENT_OF",
    "PLACE_RESIDES_IN",
    "OWNER_OF",
    "ABBREVIATION",
    "FOUNDED_BY",
    "ORGANIZES",
    "PENALIZED_AS",
]

/Users/dima/Practikum/nlp-2026/sprint_1/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
source = cast(HFDataset, load_dataset("danasone/nerel", split="train"))
print(f"Загружено документов: {len(source)}")


## EDA исходного датасета

Код Task 1–2 из `NEREL_MultiTask_Notebook.ipynb` выполняется на всех исходных документах до разбиения train/test и преобразования в `BaseNerelExample`. Сначала собираем `eda_df`, необходимый для обоих заданий.


In [ ]:
from collections import Counter
from collections.abc import Iterable
from typing import TypedDict, cast

import polars as pl


class RawNerelExample(TypedDict):
    tokens: list[str]
    tags: list[str]
    cls_vec: list[int]


# Исходные записи до преобразования в BaseNerelExample.
examples = cast(Iterable[RawNerelExample], source)

# size — число исходных слов в tokens, до субтокенизации RuBERT.
# tags — частоты BIO-меток (B- и I- учитываются отдельно, O сохраняется).
# Словари представлены нативными Struct-колонками Polars.
tag_names = sorted({tag for example in examples for tag in example["tags"]})
# Один порядок меток для всего source; сохраняем его и для будущего инференса.
label2id = {tag: index for index, tag in enumerate(tag_names)}
id2label = {index: tag for tag, index in label2id.items()}

# В источнике нет поля id: номер записи служит уникальным id документа.
rows = []
for doc_id, example in enumerate(examples):
    if len(example["tokens"]) != len(example["tags"]):
        raise ValueError(f"Документ {doc_id}: длины tokens и tags различаются")
    cls_vec = example["cls_vec"]
    if len(cls_vec) != len(CLS_LABELS) or any(
        type(value) is not int or value not in (0, 1) for value in cls_vec
    ):
        raise ValueError(
            f"Документ {doc_id}: ожидается cls_vec из 30 целых значений 0/1"
        )
    counts = Counter(example["tags"])
    rows.append(
        {
            "id": doc_id,
            "size": len(example["tokens"]),
            "tags": {tag: counts[tag] for tag in tag_names},
            "labels": dict(zip(CLS_LABELS, map(int, cls_vec), strict=True)),
        }
    )

eda_df = pl.DataFrame(
    rows,
    schema={
        "id": pl.Int64,
        "size": pl.Int64,
        "tags": pl.Struct({tag: pl.Int64 for tag in tag_names}),
        "labels": pl.Struct({label: pl.Int64 for label in CLS_LABELS}),
    },
)
if eda_df.height != len(source):
    raise ValueError("Число строк eda_df не совпадает с числом документов в source")
if eda_df["id"].n_unique() != eda_df.height:
    raise ValueError("Идентификаторы документов в eda_df не уникальны")

# Информационные метрики: длины измеряются в исходных токенах.
documents_without_labels = eda_df.filter(
    pl.all_horizontal(
        [pl.col("labels").struct.field(label) == 0 for label in CLS_LABELS]
    )
).height
max_document_size = eda_df["size"].max()
median_document_size = eda_df["size"].median()

print(f"Документов со всеми нулями в labels: {documents_without_labels}")
print(f"Максимальная длина документа в токенах: {max_document_size}")
print(f"Медианная длина документа в токенах: {median_document_size}")

eda_df.head()

### Task 1 — частоты BIO-меток


In [ ]:
# Task 1
# build facts table
# tag: str,
# id:int,
# count: int

# Допущение - каждую BIO метку анализируем независимоƒ


# calculate metrics table:
# tag: str
# documents: int - number of unique documents with this tag
# documents_share - % share of documents with this tag among all documents
# tokens: int - number of tokens with this tag
# tokens_share - % share of all tokens in dataset
# cumulative_share - %, кумулятивная метки среди токенов, доли суммировать от самой большой до самой маленькой

# token 'O' is ignored, out of scope in the data - in text tokens and tags in this analysis

# additional total metrics:
# total documents: int
# total tokens: int

# Проверки целостности:
# 1. sum(metrics_table.count) = кол-во не O токенов в исходном датасете
# 2. max(metrics_table.documents) <= кол-во документов в датасете
# 3. unique(facts_table.id) = кол-во документов в датасете
# 4. sum(metrics_table.tokens_share) = 100%
# 5. sum(facts_table.count) = кол-во не O токенов в исходном датасете
#
# Ограничения: не должно быть проверок, не относящихся к этим пунктам. Каждая проверка снабжается комментарием-номером проверки целостности в задании


# visualization - html report
# токены в диаграммах нумеровать по порядку появления на диаграмме
# накопленную долю токенов вывести на диаграмме токенов
# диаграмму токенов по частоте документов отсортировать от самых частотных меток до самых редких сверху вниз

from math import isclose

# Считаем исходные слова, не субтокены; B- и I- учитываются отдельно.
# O исключаем, а факты оставляем только для присутствующих в документе меток.
tag_facts = (
    eda_df.select("id", "tags")
    .unnest("tags")
    .unpivot(index="id", variable_name="tag", value_name="count")
    .filter((pl.col("tag") != "O") & (pl.col("count") > 0))
    .select("tag", "id", "count")
    .sort(["tag", "id"])
)

# Итоги считаем независимо от таблицы фактов, по исходному датасету.
total_documents = len(source)
total_tokens = sum(tag != "O" for example in examples for tag in example["tags"])

tag_metrics = (
    tag_facts.group_by("tag")
    .agg(
        pl.col("id").n_unique().alias("documents"),
        pl.col("count").sum().alias("tokens"),
    )
    .with_columns(
        (100.0 * pl.col("documents") / total_documents).alias("documents_share"),
        (100.0 * pl.col("tokens") / total_tokens).alias("tokens_share"),
    )
    .select("tag", "documents", "documents_share", "tokens", "tokens_share")
    .sort(["tokens", "tag"], descending=[True, False])
    .with_columns(pl.col("tokens_share").cum_sum().alias("cumulative_share"))
)

# В метриках агрегированный count называется tokens.
assert tag_metrics["tokens"].sum() == total_tokens
max_documents = tag_metrics.select(pl.col("documents").max()).item()
assert max_documents <= total_documents
assert tag_facts["id"].n_unique() == total_documents
assert isclose(tag_metrics["tokens_share"].sum(), 100.0, abs_tol=1e-9)
assert tag_facts["count"].sum() == total_tokens

In [ ]:
from runpy import run_path

from IPython.display import HTML, display

generate_tag_report = run_path("html/nerel_task1.py")["generate_tag_report"]
report_fragment, report_html, report_path = generate_tag_report(
    tag_metrics, tag_facts, total_documents, total_tokens
)
display(HTML(report_fragment))
print(f"HTML-отчёт сохранён: {report_path.resolve()}")

### Task 2 — частоты характеристик документов


In [ ]:
# Task 2 EDA labels
from runpy import run_path

from IPython.display import HTML, display

# Факт — активная характеристика документа; нулевые значения исключаем.
label_facts = (
    eda_df.select("id", "labels")
    .unnest("labels")
    .unpivot(index="id", variable_name="label", value_name="value")
    .filter(pl.col("value") == 1)
    .select("id", "label", "value")
    .sort(["id", "label"])
)

label_total_documents = len(source)
label_metrics = (
    pl.DataFrame({"label": CLS_LABELS})
    .join(
        label_facts.group_by("label").agg(pl.col("id").n_unique().alias("documents")),
        on="label",
        how="left",
    )
    .with_columns(pl.col("documents").fill_null(0))
    .with_columns(
        (100.0 * pl.col("documents") / label_total_documents).alias("documents_share")
    )
    .select("label", "documents", "documents_share")
    .sort(["documents", "label"], descending=[True, False])
)

# Проверка целостности 1: число документов с меткой не превышает размер источника.
label_max_documents = label_metrics.select(pl.col("documents").max()).item()
assert label_max_documents <= label_total_documents
# Проверка целостности 2: число уникальных документов в фактах не превышает размер источника.
assert label_facts["id"].n_unique() <= label_total_documents, (
    "Проверка 2: число уникальных документов в фактах превышает размер источника"
)
# Проверка целостности 3: в метриках ровно 30 характеристик, как в CLS_LABELS.
assert label_metrics["label"].n_unique() == len(CLS_LABELS) == 30

generate_label_report = run_path("html/nerel_task2.py")["generate_label_report"]
label_report_fragment, label_report_html, label_report_path = generate_label_report(
    label_metrics, label_facts, label_total_documents
)
display(HTML(label_report_fragment))
print(f"HTML-отчёт сохранён: {label_report_path.resolve()}")

## Датасет, справочник тегов и bundle

`NerelTagFabric` назначает стабильные id в отсортированном порядке. Датасет сохраняет обе выборки, значения и id тегов. Хранилище делегирует сохранение и загрузку датасету и фабрике через их базовые интерфейсы. В bundle нет дополнительных публичных членов.

In [2]:
class NerelTagFabric(BaseTextTagFabric):
    def __init__(self) -> None:
        self._by_id: dict[int, TextTag] = {}
        self._by_value: dict[str, TextTag] = {}

    def fit(self, tags: list[str]) -> None:
        self._by_id = {
            index: TextTag(index, value)
            for index, value in enumerate(sorted(set(tags)))
        }
        self._by_value = {tag.value: tag for tag in self._by_id.values()}

    def by_id(self, id: int) -> TextTag:
        return self._by_id[id]

    def by_value(self, value: str) -> TextTag:
        return self._by_value[value]

    def save(self, path: str) -> None:
        payload = [{"id": tag.id, "value": tag.value} for tag in self._by_id.values()]
        Path(path).write_text(json.dumps(payload, ensure_ascii=False), encoding="utf-8")

    @classmethod
    def load(cls, path: str) -> Self:
        payload = cast(
            list[dict[str, object]], json.loads(Path(path).read_text(encoding="utf-8"))
        )
        fabric = cls()
        for item in payload:
            tag = TextTag(cast(int, item["id"]), cast(str, item["value"]))
            fabric._by_id[tag.id] = tag
            fabric._by_value[tag.value] = tag
        return fabric


def _encode_example(example: BaseNerelExample) -> dict[str, object]:
    return {
        "tokens": example["tokens"].text,
        "tags": [{"id": tag.id, "value": tag.value} for tag in example["tags"]],
        "cls_vec": example["cls_vec"],
    }


def _decode_example(example: dict[str, object]) -> BaseNerelExample:
    tags = cast(list[dict[str, object]], example["tags"])
    return BaseNerelExample(
        tokens=BaseDocument(text=cast(list[TextToken], example["tokens"])),
        tags=[TextTag(cast(int, tag["id"]), cast(str, tag["value"])) for tag in tags],
        cls_vec=cast(list[TextTopic], example["cls_vec"]),
    )


class NerelDataset(BaseNerelDataset):
    def __init__(
        self, train: list[BaseNerelExample], test: list[BaseNerelExample]
    ) -> None:
        self.train = train
        self.test = test

    def save(self, path: str) -> None:
        payload = {
            "train": [_encode_example(example) for example in self.train],
            "test": [_encode_example(example) for example in self.test],
        }
        Path(path).write_text(json.dumps(payload, ensure_ascii=False), encoding="utf-8")

    @classmethod
    def load(cls, path: str) -> Self:
        payload = cast(
            dict[str, list[dict[str, object]]],
            json.loads(Path(path).read_text(encoding="utf-8")),
        )
        return cls(
            train=[_decode_example(example) for example in payload["train"]],
            test=[_decode_example(example) for example in payload["test"]],
        )


class NerelDatasetBundle(BaseNerelDatasetBundle):
    def __init__(
        self, records: BaseNerelDataset, tag_fabric: BaseTextTagFabric
    ) -> None:
        self._dataset = records
        self.tag_fabric = tag_fabric

    def dataset(self) -> BaseNerelDataset:
        return self._dataset


class DatasetStorage(BaseDatasetStorage):
    def __init__(
        self,
        dataset_type: type[BaseNerelDataset],
        fabric_type: type[BaseTextTagFabric],
    ) -> None:
        self._dataset_type = dataset_type
        self._fabric_type = fabric_type

    def save(self, path: str, dataset: BaseNerelDatasetBundle) -> None:
        target = Path(path)
        target.mkdir(parents=True, exist_ok=True)
        dataset.dataset().save(str(target / "dataset.json"))
        dataset.tag_fabric.save(str(target / "tags.json"))

    def load(self, path: str) -> BaseNerelDatasetBundle:
        source = Path(path)
        return NerelDatasetBundle(
            self._dataset_type.load(str(source / "dataset.json")),
            self._fabric_type.load(str(source / "tags.json")),
        )

In [ ]:
source_splits = source.train_test_split(test_size=TEST_SIZE, seed=SEED)
train_rows = list(cast(Iterable[dict[str, object]], source_splits["train"]))
test_rows = list(cast(Iterable[dict[str, object]], source_splits["test"]))

tag_fabric: BaseTextTagFabric = NerelTagFabric()
tag_fabric.fit([tag for row in train_rows for tag in cast(list[str], row["tags"])])


def prepare_examples(
    rows: list[dict[str, object]], fabric: BaseTextTagFabric
) -> list[BaseNerelExample]:
    return [
        BaseNerelExample(
            tokens=BaseDocument(text=cast(list[TextToken], row["tokens"])),
            tags=[fabric.by_value(tag) for tag in cast(list[str], row["tags"])],
            cls_vec=cast(list[TextTopic], row["cls_vec"]),
        )
        for row in rows
    ]


bundle: BaseNerelDatasetBundle = NerelDatasetBundle(
    NerelDataset(
        prepare_examples(train_rows, tag_fabric),
        prepare_examples(test_rows, tag_fabric),
    ),
    tag_fabric,
)
print(f"Train: {len(bundle.dataset().train)}; test: {len(bundle.dataset().test)}")

## Сохранение и восстановление

По заданию проверяем, что обе выборки и соответствие id/значение тегов полностью восстановились. При повторном запуске перезаписывается только папка `artifacts/nerel_reorganized`. Дальше используются данные и фабрика из загруженного bundle.

In [4]:
storage: BaseDatasetStorage = DatasetStorage(NerelDataset, NerelTagFabric)
storage.save(str(BUNDLE_PATH), bundle)
restored_bundle = storage.load(str(BUNDLE_PATH))

# Проверка сохранения и загрузки bundle, явно указанная в Task 1.
assert restored_bundle.dataset().train == bundle.dataset().train
assert restored_bundle.dataset().test == bundle.dataset().test
for example in bundle.dataset().train + bundle.dataset().test:
    for tag in example["tags"]:
        assert restored_bundle.tag_fabric.by_id(tag.id) == tag
        assert restored_bundle.tag_fabric.by_value(tag.value) == tag

print(f"Bundle восстановлен: {BUNDLE_PATH.resolve()}")
print("Обе выборки и id/значения тегов совпадают с сохранёнными.")

Bundle восстановлен: /Users/dima/Practikum/nlp-2026/sprint_1/src/project/artifacts/nerel_reorganized
Обе выборки и id/значения тегов совпадают с сохранёнными.


## Две модели

`infer(documents: list[BaseDocument])` принимает список документов и возвращает по одному `BaseTextPrediction` на документ в исходном порядке. Поле `text` каждого документа содержит список слов; пустой вход даёт пустой список предсказаний. Случайная модель равномерно выбирает теги, включая `O`, и независимо генерирует 0/1 для каждой характеристики. Постоянная модель ставит `O` каждому слову и возвращает одинаковый multihot-вектор: единицы у `WORKPLACE`, `ALTERNATIVE_NAME`, `WORKS_AS`, `PARTICIPANT_IN` (как в предыдущем ноутбуке).

Обе модели реализуют `save()`/`load()`. Для случайной модели сохраняется и состояние генератора. Модели получают только список `BaseDocument` со словами; истинные метки используются при вычислении метрик.

In [5]:
class RandomModel(BaseModel):
    def __init__(self, tags: list[TextTag], topic_count: int, seed: int = 42) -> None:
        self._tags = tags.copy()
        self._topic_count = topic_count
        self._random = Random(seed)

    def infer(self, documents: list[BaseDocument]) -> list[BaseTextPrediction]:
        return [
            BaseTextPrediction(
                input_text=BaseDocument(text=document.text.copy()),
                tag_prediction=[self._random.choice(self._tags) for _ in document.text],
                topic_prediction=[
                    self._random.choice((0, 1)) for _ in range(self._topic_count)
                ],
            )
            for document in documents
        ]

    def save(self, path: str) -> None:
        Path(path).write_text(
            json.dumps(
                {
                    "tags": [{"id": tag.id, "value": tag.value} for tag in self._tags],
                    "topic_count": self._topic_count,
                    "random_state": self._random.getstate(),
                },
                ensure_ascii=False,
            ),
            encoding="utf-8",
        )

    def load(self, path: str) -> None:
        payload = cast(
            dict[str, object], json.loads(Path(path).read_text(encoding="utf-8"))
        )
        self._tags = [
            TextTag(cast(int, tag["id"]), cast(str, tag["value"]))
            for tag in cast(list[dict[str, object]], payload["tags"])
        ]
        self._topic_count = cast(int, payload["topic_count"])
        version, state, gauss = cast(
            tuple[int, list[int], float | None], payload["random_state"]
        )
        self._random.setstate((version, tuple(state), gauss))


class ConstantModel(BaseModel):
    def __init__(self, tag: TextTag, topics: list[TextTopic]) -> None:
        self._tag = tag
        self._topics = topics.copy()

    def infer(self, documents: list[BaseDocument]) -> list[BaseTextPrediction]:
        return [
            BaseTextPrediction(
                input_text=BaseDocument(text=document.text.copy()),
                tag_prediction=[self._tag for _ in document.text],
                topic_prediction=self._topics.copy(),
            )
            for document in documents
        ]

    def save(self, path: str) -> None:
        Path(path).write_text(
            json.dumps(
                {
                    "tag": {"id": self._tag.id, "value": self._tag.value},
                    "topics": self._topics,
                },
                ensure_ascii=False,
            ),
            encoding="utf-8",
        )

    def load(self, path: str) -> None:
        payload = cast(
            dict[str, object], json.loads(Path(path).read_text(encoding="utf-8"))
        )
        tag = cast(dict[str, object], payload["tag"])
        self._tag = TextTag(cast(int, tag["id"]), cast(str, tag["value"]))
        self._topics = cast(list[TextTopic], payload["topics"])

## Метрики тегов и характеристик

Повторяем метрики исходного ноутбука: `f1-macro-tokens`, `precision-tokens`, `recall-tokens` и соответствующие три метрики `topics`. Все используют sklearn, macro-усреднение и `zero_division=0`.

NER оценивается по исходным словам, по всему справочнику BIO-тегов без `O`. Правило всех BIO-метрик: перед расчётом синхронно удаляем из ground truth и предсказаний только пары `(O, O)`. `O` также исключён из оцениваемых классов. Пары `(O, сущность)` и `(сущность, O)` сохраняются как false positive и false negative соответственно; оставшиеся пары относятся к одним и тем же исходным словам. Если после фильтрации пар нет, все три метрики равны 0. Это классификация токенов; точное совпадение целых сущностей не оценивается. CLS оценивается как матрица «документы × 30 независимых бинарных характеристик» по всем 30 колонкам.

In [6]:
def metric_targets(
    y_true: list[BaseNerelExample],
    y_pred: list[BaseTextPrediction],
    target: Literal["tokens", "topics"],
) -> tuple[list[int] | list[list[int]], list[int] | list[list[int]]]:
    if target == "tokens":
        # Одна маска для обеих сторон сохраняет соответствие исходным словам.
        pairs = [
            (actual.id, predicted.id)
            for example, prediction in zip(y_true, y_pred)
            for actual, predicted in zip(example["tags"], prediction["tag_prediction"])
            if actual.value != "O" or predicted.value != "O"
        ]
        return (
            [actual for actual, _ in pairs],
            [predicted for _, predicted in pairs],
        )
    return (
        [example["cls_vec"] for example in y_true],
        [prediction["topic_prediction"] for prediction in y_pred],
    )


class MacroMetric(BaseMetric):
    def __init__(
        self,
        name: str,
        scorer: Callable[..., float],
        label_ids: list[int],
        target: Literal["tokens", "topics"],
    ) -> None:
        self._name = name
        self._scorer = scorer
        self._label_ids = label_ids.copy()
        self._target: Literal["tokens", "topics"] = target

    def eval(
        self, y_true: list[BaseNerelExample], y_pred: list[BaseTextPrediction]
    ) -> float:
        actual, predicted = metric_targets(y_true, y_pred, self._target)
        if self._target == "tokens" and not actual:
            return 0.0
        return float(
            self._scorer(
                actual,
                predicted,
                labels=self._label_ids,
                average="macro",
                zero_division=0,
            )
        )

    def get_name(self) -> str:
        return f"{self._name}-{self._target}"


class MetricEvaluator(BaseMetricEvaluator):
    def __init__(self) -> None:
        self._metrics: list[BaseMetric] = []

    def add(self, metric: BaseMetric) -> None:
        self._metrics.append(metric)

    def eval(
        self, y_true: list[BaseNerelExample], y_pred: list[BaseTextPrediction]
    ) -> dict[str, float]:
        return {
            metric.get_name(): metric.eval(y_true, y_pred) for metric in self._metrics
        }

In [7]:
records = restored_bundle.dataset()
fabric = restored_bundle.tag_fabric
# В интерфейсе фабрики нет перечисления; имена берём из восстановленного train.
tag_values = sorted({tag.value for example in records.train for tag in example["tags"]})
evaluation_tags = [fabric.by_value(value) for value in tag_values]

# Общее правило BIO-метрик: O не участвует в усреднении.
# Полный справочник evaluation_tags сохраняется для предсказаний моделей.
metric_tag_ids = [tag.id for tag in evaluation_tags if tag.value != "O"]

constant_topic_names = {"WORKPLACE", "ALTERNATIVE_NAME", "WORKS_AS", "PARTICIPANT_IN"}
constant_topics = [int(name in constant_topic_names) for name in CLS_LABELS]
models: dict[str, BaseModel] = {
    "RandomModel": RandomModel(evaluation_tags, len(CLS_LABELS), seed=SEED),
    "ConstantModel": ConstantModel(fabric.by_value("O"), constant_topics),
}

token_evaluator: BaseMetricEvaluator = MetricEvaluator()
topic_evaluator: BaseMetricEvaluator = MetricEvaluator()
for name, scorer in (
    ("f1-macro", f1_score),
    ("precision", precision_score),
    ("recall", recall_score),
):
    token_evaluator.add(MacroMetric(name, scorer, metric_tag_ids, target="tokens"))
    topic_evaluator.add(
        MacroMetric(name, scorer, list(range(len(CLS_LABELS))), target="topics")
    )

results: dict[str, dict[str, float]] = {}
print(
    f"Test: {len(records.test)} документов; "
    f"{sum(len(example['tokens'].text) for example in records.test):,} слов"
)
print(
    f"NER: {len(metric_tag_ids)} BIO-тегов без O; CLS: {len(CLS_LABELS)} характеристик"
)
for model_name, model in models.items():
    predictions = model.infer([example["tokens"] for example in records.test])
    results[model_name] = {
        **token_evaluator.eval(records.test, predictions),
        **topic_evaluator.eval(records.test, predictions),
    }
    print(f"\n{model_name}")
    for metric_name, value in results[model_name].items():
        print(f"{metric_name:24s} {value:.6f}")

Test: 150 документов; 31,863 слов
NER: 57 BIO-тегов без O; CLS: 30 характеристик

RandomModel
f1-macro-tokens          0.008119
precision-tokens         0.008120
recall-tokens            0.020837
f1-macro-topics          0.402502
precision-topics         0.418699
recall-topics            0.506763

ConstantModel
f1-macro-tokens          0.000000
precision-tokens         0.000000
recall-tokens            0.000000
f1-macro-topics          0.127910
precision-topics         0.123111
recall-topics            0.133333


Обе модели служат базовыми ориентирами и не обучаются. Постоянный тег `O` даёт нулевую полноту для сущностей; постоянные положительные CLS-характеристики имеют recall 1 при наличии положительных примеров, остальные — 0. Macro-усреднение учитывает все метки, а не только выбранные постоянной моделью.

На этом Task 1 завершён.

Task 2: implement html visualizer of model predictions.
This should be a page of two blocks:
first block comparizon of ground truth topic vs predicted topics
second block - one text block of the input text with highlights of ground truth and predicted tags
implement this 
and visualize one random example per constant and random model


# Task 2 — HTML-визуализация предсказаний

`HtmlPredictionVisualizer` в новом файле `html/prediction_visualizer.py` реализует `BaseVisualizer`. Первая секция сравнивает характеристики документа: истину, прогноз и расхождения. Характеристики с нулём и в истине, и в прогнозе скрыты. Вторая показывает текст один раз: верхняя половина подсветки каждого слова соответствует истинному тегу, нижняя — предсказанному. Точные BIO-теги подписаны сверху и снизу; тип сущности имеет один цвет для B- и I-, а несовпадения отмечены пунктиром.

`BaseDocument` без эталонной разметки также поддерживается: вместо истины выводится «—». Исходные пробелы недоступны в датасете, поэтому слова соединяются пробелами.

Для каждой модели выбираем случайный документ из test с отдельным генератором и seed 42. HTML отображается непосредственно в ноутбуке; отдельные файлы отчётов не создаются. Существующие HTML-скрипты относятся к другому контексту и не изменяются.

В каждой секции показаны precision, recall и F1 для текущего документа. Визуализатор получает существующие `token_evaluator` и `topic_evaluator` через интерфейс `BaseMetricEvaluator` и вызывает их `eval([example], [prediction])`. Используются реализации `MetricEvaluator` и `MacroMetric` из Task 1: macro по всему справочнику BIO-тегов (без O) и по всем 30 характеристикам, `zero_division=0`. Скрытие строк таблицы не меняет расчёт метрик. Без эталонной разметки метрики недоступны.

Перед BIO-метриками из ground truth и предсказаний синхронно удаляются только пары `(O, O)`; пары `(O, сущность)` и `(сущность, O)` учитываются как ошибки. Это общее поведение `metric_targets`, используемого всеми тремя метриками; при пустом остатке возвращается 0.

Третий пример использует ещё один экземпляр существующего `ConstantModel`: тег каждого слова — `B-PERSON`; положительные характеристики — `ALTERNATIVE_NAME`, `WORKS_AS`, `PARTICIPANT_IN`, `WORKPLACE`, `POINT_IN_TIME`, `TAKES_PLACE_IN`, `AGE_IS`. Остальные характеристики равны 0.


In [8]:
from runpy import run_path

from base import BaseVisualizer

visualizer_type = cast(
    Callable[
        [list[str], str, BaseMetricEvaluator, BaseMetricEvaluator], BaseVisualizer
    ],
    run_path("html/prediction_visualizer.py")["HtmlPredictionVisualizer"],
)
person_topic_names = {
    "ALTERNATIVE_NAME",
    "WORKS_AS",
    "PARTICIPANT_IN",
    "WORKPLACE",
    "POINT_IN_TIME",
    "TAKES_PLACE_IN",
    "AGE_IS",
}
visualization_models: dict[str, BaseModel] = {
    "ConstantModel": models["ConstantModel"],
    "RandomModel": models["RandomModel"],
    "ConstantModel B-PERSON": ConstantModel(
        fabric.by_value("B-PERSON"),
        [int(name in person_topic_names) for name in CLS_LABELS],
    ),
}
example_random = Random(SEED)
for model_name, visualization_model in visualization_models.items():
    example_index = example_random.randrange(len(records.test))
    example = records.test[example_index]
    prediction = visualization_model.infer([example["tokens"]])[0]
    visualizer = visualizer_type(
        CLS_LABELS,
        f"{model_name} · test, документ {example_index}",
        token_evaluator,
        topic_evaluator,
    )
    visualizer.visualize(example, prediction)

Характеристика,Истина,Прогноз,Сравнение
WORKPLACE,Нет,Да,Лишняя
ALTERNATIVE_NAME,Да,Да,Совпадает
WORKS_AS,Да,Да,Совпадает
PARTICIPANT_IN,Да,Да,Совпадает
POINT_IN_TIME,Да,Нет,Пропущена
TAKES_PLACE_IN,Да,Нет,Пропущена
HEADQUARTERED_IN,Да,Нет,Пропущена
ORIGINS_FROM,Да,Нет,Пропущена
AGE_IS,Да,Нет,Пропущена
PRODUCES,Да,Нет,Пропущена


Характеристика,Истина,Прогноз,Сравнение
ALTERNATIVE_NAME,Да,Нет,Пропущена
WORKS_AS,Да,Да,Совпадает
PARTICIPANT_IN,Да,Нет,Пропущена
TAKES_PLACE_IN,Да,Да,Совпадает
HEADQUARTERED_IN,Да,Нет,Пропущена
AGE_IS,Да,Да,Совпадает
HAS_CAUSE,Нет,Да,Лишняя
PRODUCES,Нет,Да,Лишняя
MEMBER_OF,Да,Нет,Пропущена
CONVICTED_OF,Нет,Да,Лишняя


Характеристика,Истина,Прогноз,Сравнение
WORKPLACE,Да,Да,Совпадает
ALTERNATIVE_NAME,Да,Да,Совпадает
WORKS_AS,Да,Да,Совпадает
PARTICIPANT_IN,Да,Да,Совпадает
POINT_IN_TIME,Да,Да,Совпадает
TAKES_PLACE_IN,Да,Да,Совпадает
HEADQUARTERED_IN,Да,Нет,Пропущена
ORIGINS_FROM,Да,Нет,Пропущена
LOCATED_IN,Да,Нет,Пропущена
AGENT,Да,Нет,Пропущена


Task 3:
look into docs/subword_model.mmd
implement BaseBatchedModel and its associations
BaseSubwordToWordAligner - this should take subword tokens from prediction, subword tags and in-sample mapping get_word_ids to map subword predicted tags to word tags. Word tag - tag of a first subword token

store classes in a new package batch/

Don't touch BaseTokenizedNerelExample, BaseBertModelLabeledInput

here create an instance of BaseBatchModel with rubert tiny backbone, two heads for token tag prediction and text topic prediction prediction, don't train it. take one sample from the test, instaciate BatchModelAdapter and call it for prediction, visualize the result with visualized.
Long documents just shrink to a maximum length of the model. Aligner should pad predicted tags with tag O to reach the input tag length

## Task 3 — RuBERT tiny, субтокены и адаптер

Классы находятся в `batch/`; имя `BaseBatchModel` соответствует диаграмме. `BaseTokenizedNerelExample` и `BaseBertModelLabeldInput` не изменяются. Поле `encoder: torch.nn.Module` реализовано по явному требованию диаграммы.

Используем [cointegrated/rubert-tiny](https://huggingface.co/cointegrated/rubert-tiny): предобученный backbone и две новые линейные головы. NER выбирает argmax для каждого субтокена, CLS независимо порогует логиты 30 характеристик по нулю (sigmoid ≥ 0.5). Головы не обучены, поэтому качество предсказаний не характеризует обученную модель.

`BatchModelAdapter.infer(documents: list[BaseDocument])` принимает список документов и возвращает предсказания в том же порядке. Каждый документ токенизируется и выравнивается отдельно, чтобы локальные `word_ids` разных документов не смешивались. Адаптер вызывает `BaseSubwordToWordAligner.align(tokenized, prediction, [document])`: параметр `input: list[BaseDocument]` содержит один документ, результат — список с одним предсказанием. Batch-токенизатор принимает `list[BaseDocument]` и передаёт списки `document.text` fast-токенизатору с `is_split_into_words=True`; каждому документу соответствует одно окно, длинный документ обрезается до максимальной длины модели. `get_word_ids(batch_index)` возвращает индексы слов исходного документа; специальные токены и padding пропускаются. `BaseSubwordToWordAligner` берёт тег первого встреченного субтокена каждого слова, включая частично попавшее в окно слово. Для слов, не покрытых предсказанием модели, алайнер ставит `O`, сохраняя длину исходного документа. BIO-теги не исправляются. Характеристики документа берутся из предсказания его единственного окна.

Выполняем только инференс одного случайного документа test с seed 42 и отображаем результат существующим HTML-визуализатором. Отдельные файлы предсказаний и отчётов не создаются.


In [9]:
import torch
from transformers import (
    AutoModel,
    AutoTokenizer,
    PreTrainedModel,
    PreTrainedTokenizerFast,
)

from batch import (
    BaseBatchModel,
    BaseBatchTokenizer,
    BaseModelType,
    BaseSubwordToWordAligner,
    BatchModelAdapter,
    HuggingFaceBatchTokenizer,
    RubertBatchModel,
)

rubert_model_type = BaseModelType(model_name="cointegrated/rubert-tiny")
rubert_encoder = cast(
    PreTrainedModel,
    AutoModel.from_pretrained(rubert_model_type.model_name, add_pooling_layer=False),
)
rubert_hf_tokenizer = cast(
    PreTrainedTokenizerFast,
    AutoTokenizer.from_pretrained(rubert_model_type.model_name, use_fast=True),
)
rubert_tokenizer: BaseBatchTokenizer = HuggingFaceBatchTokenizer(
    rubert_model_type,
    rubert_hf_tokenizer,
    max_length=int(rubert_encoder.config.max_position_embeddings),
)
# Фиксируем инициализацию голов, не меняя состояние генератора других моделей.
with torch.random.fork_rng(devices=[]):
    torch.manual_seed(SEED)
    rubert_batch_model: BaseBatchModel = RubertBatchModel(
        rubert_model_type,
        rubert_encoder,
        num_token_tags=len(evaluation_tags),
        num_topics=len(CLS_LABELS),
    )
rubert_adapter: BaseModel = BatchModelAdapter(
    rubert_batch_model,
    rubert_tokenizer,
    BaseSubwordToWordAligner(fabric),
)

Loading weights: 100%|██████████| 53/53 [00:00<00:00, 15048.61it/s]
[transformers] BertModel LOAD REPORT from: cointegrated/rubert-tiny
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
bert.pooler.dense.weight                   | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
bert.pooler.dense.bias                     | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [10]:
rubert_example_index = Random(SEED).randrange(len(records.test))
rubert_example = records.test[rubert_example_index]
rubert_prediction = rubert_adapter.infer([rubert_example["tokens"]])[0]
print(
    f"{rubert_model_type.model_name} · test, документ {rubert_example_index} · "
    f"{len(rubert_example['tokens'].text)} слов · головы без обучения"
)
rubert_visualizer = visualizer_type(
    CLS_LABELS,
    f"RuBERT tiny · головы без обучения · test, документ {rubert_example_index}",
    token_evaluator,
    topic_evaluator,
)
rubert_visualizer.visualize(rubert_example, rubert_prediction)

cointegrated/rubert-tiny · test, документ 28 · 184 слов · головы без обучения


Характеристика,Истина,Прогноз,Сравнение
WORKPLACE,Нет,Да,Лишняя
ALTERNATIVE_NAME,Да,Да,Совпадает
WORKS_AS,Да,Нет,Пропущена
PARTICIPANT_IN,Да,Нет,Пропущена
POINT_IN_TIME,Да,Да,Совпадает
TAKES_PLACE_IN,Да,Нет,Пропущена
HEADQUARTERED_IN,Да,Да,Совпадает
ORIGINS_FROM,Да,Нет,Пропущена
AGENT,Нет,Да,Лишняя
AGE_IS,Да,Да,Совпадает


Task 4:
Implement BaseLabledTokenizedBatch, BaseBatchLabler from docs/subword_model.mmd in batch/

build here pytorch dataset train/test from existing split, use labler and tokenizer to collect labled batches from orignal dataset
build here dataloader
run rubert batch model with dataloader without training

## Task 4 — размеченные батчи и DataLoader

`BaseLabeledTokenizedBatch` (имя из диаграммы) наследует `BaseBatchTokenized`: сохраняет вход модели, `get_word_ids`, NER-цели `tags` и характеристики `cls_vec`. `BaseBatchLabler` переносит id тега слова на его первый субтокен. Специальные токены, padding и продолжения слов получают `-100` для исключения из будущего token loss. Обрезанные слова не добавляются в субтокенную разметку; CLS-вектор документа сохраняется целиком.

PyTorch Dataset возвращает исходные примеры существующих train/test без нового разбиения. `collate_fn` токенизирует документы совместно с динамическим padding и собирает размеченный батч. DataLoader использует `num_workers=0`; train перемешивается с seed 42, test сохраняет порядок. Вход остаётся списками согласно интерфейсу диаграммы; модель сама создаёт тензоры.

Проходим обе выборки через существующий `rubert_batch_model.forward`. Разметка не участвует в инференсе; loss, backward и обновление весов не выполняются. Предсказания не накапливаются в памяти. Головы остаются необученными.


In [11]:
from torch.utils.data import DataLoader, Dataset

from base import BaseNerelExample
from batch import BaseBatchLabler, BaseLabeledTokenizedBatch


class NerelTorchDataset(Dataset[BaseNerelExample]):
    def __init__(self, examples: list[BaseNerelExample]) -> None:
        self._examples = examples

    def __len__(self) -> int:
        return len(self._examples)

    def __getitem__(self, index: int) -> BaseNerelExample:
        return self._examples[index]


batch_labler = BaseBatchLabler()


def collate_nerel_batch(
    examples: list[BaseNerelExample],
) -> BaseLabeledTokenizedBatch:
    tokenized = rubert_tokenizer.tokenize([example["tokens"] for example in examples])
    return batch_labler.build_batch(examples, tokenized)


train_dataset = NerelTorchDataset(records.train)
test_dataset = NerelTorchDataset(records.test)
BATCH_SIZE = 8
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    generator=torch.Generator().manual_seed(SEED),
    collate_fn=collate_nerel_batch,
    num_workers=0,
)
test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=collate_nerel_batch,
    num_workers=0,
)
print(f"Train: {len(train_dataset)} документов, {len(train_loader)} батчей")
print(f"Test: {len(test_dataset)} документов, {len(test_loader)} батчей")

Train: 596 документов, 75 батчей
Test: 150 документов, 19 батчей


In [12]:
for split_name, loader in (("train", train_loader), ("test", test_loader)):
    document_count = 0
    batch_count = 0
    labeled_subtoken_count = 0
    for labeled_batch in loader:
        batch_prediction = rubert_batch_model.forward(labeled_batch)
        document_count += len(batch_prediction["text_classes"])
        batch_count += 1
        labeled_subtoken_count += sum(
            tag_id != -100 for row in labeled_batch.tags for tag_id in row
        )
    print(
        f"{split_name}: инференс {document_count} документов, {batch_count} батчей; "
        f"NER-целей на первых субтокенах: {labeled_subtoken_count:,}; без обучения"
    )

train: инференс 596 документов, 75 батчей; NER-целей на первых субтокенах: 115,439; без обучения
test: инференс 150 документов, 19 батчей; NER-целей на первых субтокенах: 28,148; без обучения


### Task 5

Implement train loop
instantiate train_loader, test_loader with batch_size as parameter
define loss:
take loss = CrossEntropyLoss(token_logits) + BCELoss(topic_logits)
class MultitaskLoss(nn.Module):
    def __init__(self, use_uncertainty_weight=False) -> None:
        super().__init__()
        self._token_loss = nn.CrossEntropyLoss(
            ignore_index=IGNORE_LABEL, reduction=TASK8_LOSS_REDUCTION
        )
        self._topic_loss = nn.BCELoss(reduction=TASK8_LOSS_REDUCTION)
        self._loss: torch.Tensor
        self.log_sigma_token = nn.Parameter(torch.tensor(0.0))
        self.log_sigma_topic = nn.Parameter(torch.tensor(0.0))
        self.use_uncertainty_weight = use_uncertainty_weight

    def forward(
        self,
        token_logits: torch.Tensor,
        token_labels: torch.Tensor,
        topic_logits: torch.Tensor,
        topic_labels: torch.Tensor,
    ) -> torch.Tensor:
        valid_tokens = token_labels != IGNORE_LABEL
        token_loss = self._token_loss(
            token_logits[valid_tokens], token_labels[valid_tokens]
        ) / valid_tokens.sum().clamp_min(TASK8_MIN_COUNT)
        topic_loss = (
            self._topic_loss(topic_logits.sigmoid(), topic_labels)
            / topic_labels.numel()
        )
        if self.use_uncertainty_weight:
            loss_token_term = torch.exp(-2.0 * self.log_sigma_token) * token_loss + self.log_sigma_token
            loss_topic_term = torch.exp(-2.0 * self.log_sigma_topic) * topic_loss + self.log_sigma_topic
            loss = loss_token_term + loss_topic_term
        else:
            loss = token_loss + topic_loss 
        self._loss = loss
        return self._loss


use AdamOptimizer lr = 1e-3
epochs = 5





strictly use this function signatures:


def eval_model(model: ModelAdapter, loader) -> prints metrics report 

ModelAdapter - model adapter from task 7

def train_epoch_model(model, loss: MultiTaskLoss, optimizer)


call train/eval loop for epochs
after every epoch log train loss, train f1 on tokens, test f1 on tokens, train f1 on topics, test f1 on topics

## Task 5 — обучение NER и характеристик документа

`RubertBatchModel` теперь также является `nn.Module`: `logits` вычисляет логиты батча с градиентами, а прежний `forward` сохраняет поведение инференса. Adam обновляет backbone и обе головы. `BATCH_SIZE` задаёт размер обоих DataLoader; train перемешивается, test — нет.

Loss — сумма CE по первым субтокенам (остальные позиции `-100`) и BCE по sigmoid логитов характеристик. Обе части нормируются отдельно; опциональные uncertainty-веса по умолчанию выключены. При включении их параметры тоже передаются Adam.

В условии нет соответствующего Task 7, поэтому `ModelAdapter` — псевдоним существующего `BatchModelAdapter` из Task 3, а `MultiTaskLoss` — псевдоним заданного `MultitaskLoss`. Сигнатуры `eval_model(model, loader)` и `train_epoch_model(model, loss, optimizer)` сохранены; функция обучения использует `train_loader` текущего эксперимента.

После каждой эпохи печатаем средний train loss и отчёты train/test с F1, precision и recall для NER и характеристик. Метрики считаются по исходным словам через адаптер и evaluator из Task 1: BIO без O, синхронное удаление пар `(O, O)`, дополнение обрезанного хвоста тегом O. Train-метрики измеряются после завершения эпохи, с выключенным dropout. Файлы отчётов и чекпойнты не создаются.

Task 5 перезагружает модуль модели. Непосредственно перед циклом обучения `model_name` задаёт pretrained backbone и соответствующий токенизатор. Повторный запуск загружает исходные веса backbone и заново инициализирует головы; DataLoader использует токенизатор выбранной модели.

Обучение и оценка показывают прогресс через `tqdm`. Для Task 5 collate возвращает токенизированный батч вместе с исходными примерами. `eval_model` перебирает DataLoader, вызывает `infer_batch` адаптера на каждом готовом батче и собирает общие метрики по выборке. Исходные примеры сохраняют соответствие при shuffle и позволяют учитывать обрезанный хвост документа.


In [13]:
from importlib import reload
from itertools import chain

from torch import nn
from torch.optim import Adam, Optimizer
from tqdm.auto import tqdm

from batch import adapter as batch_adapter_module
from batch import model as batch_model_module
from batch.adapter import BatchModelAdapter as ModelAdapter

# Обновляем класс, даже если batch импортирован в ядре до Task 5.
reload(batch_model_module)
reload(batch_adapter_module)

IGNORE_LABEL = -100
TASK8_LOSS_REDUCTION = "sum"
TASK8_MIN_COUNT = 1


class MultitaskLoss(nn.Module):
    def __init__(self, use_uncertainty_weight=False) -> None:
        super().__init__()
        self._token_loss = nn.CrossEntropyLoss(
            ignore_index=IGNORE_LABEL, reduction=TASK8_LOSS_REDUCTION
        )
        self._topic_loss = nn.BCELoss(reduction=TASK8_LOSS_REDUCTION)
        self._loss: torch.Tensor
        self.log_sigma_token = nn.Parameter(torch.tensor(0.0))
        self.log_sigma_topic = nn.Parameter(torch.tensor(0.0))
        self.use_uncertainty_weight = use_uncertainty_weight

    def forward(
        self,
        token_logits: torch.Tensor,
        token_labels: torch.Tensor,
        topic_logits: torch.Tensor,
        topic_labels: torch.Tensor,
    ) -> torch.Tensor:
        valid_tokens = token_labels != IGNORE_LABEL
        token_loss = self._token_loss(
            token_logits[valid_tokens], token_labels[valid_tokens]
        ) / valid_tokens.sum().clamp_min(TASK8_MIN_COUNT)
        topic_loss = (
            self._topic_loss(topic_logits.sigmoid(), topic_labels)
            / topic_labels.numel()
        )
        if self.use_uncertainty_weight:
            loss_token_term = (
                torch.exp(-2.0 * self.log_sigma_token) * token_loss
                + self.log_sigma_token
            )
            loss_topic_term = (
                torch.exp(-2.0 * self.log_sigma_topic) * topic_loss
                + self.log_sigma_topic
            )
            loss = loss_token_term + loss_topic_term
        else:
            loss = token_loss + topic_loss
        self._loss = loss
        return self._loss


MultiTaskLoss = MultitaskLoss


def collate_training_batch(
    examples: list[BaseNerelExample],
) -> tuple[BaseLabeledTokenizedBatch, list[BaseNerelExample]]:
    # Сохраняем исходную разметку именно этого батча, в том числе при shuffle.
    return collate_nerel_batch(examples), examples


def make_loaders(batch_size: int):
    return (
        DataLoader(
            train_dataset,
            batch_size=batch_size,
            shuffle=True,
            generator=torch.Generator().manual_seed(SEED),
            collate_fn=collate_training_batch,
            num_workers=0,
        ),
        DataLoader(
            test_dataset,
            batch_size=batch_size,
            shuffle=False,
            collate_fn=collate_training_batch,
            num_workers=0,
        ),
    )


def train_epoch_model(
    model: RubertBatchModel, loss: MultiTaskLoss, optimizer: Optimizer
) -> float:
    model.train()
    loss.train()
    device = next(model.parameters()).device
    total_loss = 0.0
    document_count = 0
    progress = tqdm(train_loader, desc="Train", unit="batch", leave=False)
    for labeled_batch, _ in progress:
        optimizer.zero_grad(set_to_none=True)
        token_logits, topic_logits = model.logits(labeled_batch)
        token_labels = torch.tensor(labeled_batch.tags, dtype=torch.long, device=device)
        topic_labels = torch.tensor(
            labeled_batch.cls_vec, dtype=torch.float32, device=device
        )
        batch_loss = loss(token_logits, token_labels, topic_logits, topic_labels)
        batch_loss.backward()
        optimizer.step()
        count = len(labeled_batch.cls_vec)
        total_loss += batch_loss.detach().item() * count
        document_count += count
        progress.set_postfix(loss=f"{total_loss / document_count:.4f}")
    return total_loss / max(document_count, 1)


def eval_model(
    model: ModelAdapter,
    loader: DataLoader[BaseNerelExample],
) -> None:
    examples: list[BaseNerelExample] = []
    predictions: list[BaseTextPrediction] = []
    for labeled_batch, batch_examples in tqdm(
        loader, desc="Eval", unit="batch", leave=False
    ):
        predictions.extend(
            model.infer_batch(
                labeled_batch, [example["tokens"] for example in batch_examples]
            )
        )
        examples.extend(batch_examples)
    report = {
        **token_evaluator.eval(examples, predictions),
        **topic_evaluator.eval(examples, predictions),
    }
    for metric_name, value in report.items():
        print(f"{metric_name:24s} {value:.6f}")

In [ ]:
model_name = "cointegrated/rubert-tiny"
BATCH_SIZE = 8
EPOCHS = 5
LEARNING_RATE = 1e-3
USE_UNCERTAINTY_WEIGHT = False

rubert_model_type = BaseModelType(model_name=model_name)
rubert_encoder = cast(
    PreTrainedModel,
    AutoModel.from_pretrained(model_name, add_pooling_layer=False),
)
rubert_hf_tokenizer = cast(
    PreTrainedTokenizerFast,
    AutoTokenizer.from_pretrained(model_name, use_fast=True),
)
rubert_tokenizer = HuggingFaceBatchTokenizer(
    rubert_model_type,
    rubert_hf_tokenizer,
    max_length=int(rubert_encoder.config.max_position_embeddings),
)
train_loader, test_loader = make_loaders(batch_size=BATCH_SIZE)
# Для каждого запуска загружаем pretrained backbone и создаём новые головы.
with torch.random.fork_rng(devices=[]):
    torch.manual_seed(SEED)
    trainable_model = batch_model_module.RubertBatchModel(
        rubert_model_type,
        rubert_encoder,
        num_token_tags=len(evaluation_tags),
        num_topics=len(CLS_LABELS),
    )
rubert_batch_model = trainable_model
training_adapter = batch_adapter_module.BatchModelAdapter(
    trainable_model, rubert_tokenizer, BaseSubwordToWordAligner(fabric)
)
multitask_loss = MultitaskLoss(use_uncertainty_weight=USE_UNCERTAINTY_WEIGHT).to(
    next(trainable_model.parameters()).device
)
optimizer = Adam(
    chain(trainable_model.parameters(), multitask_loss.parameters()),
    lr=LEARNING_RATE,
)

for epoch in tqdm(range(1, EPOCHS + 1), desc="Epochs", unit="epoch"):
    train_loss = train_epoch_model(trainable_model, multitask_loss, optimizer)
    print(f"\nEpoch {epoch}/{EPOCHS} · train loss: {train_loss:.6f}")
    print("Train")
    eval_model(training_adapter, train_loader)
    print("Test")
    eval_model(training_adapter, test_loader)

Task 6:
call visualizer to show one random prediction of batched model with rubert backbone from test set

In [15]:
rubert_example = records.test[rubert_example_index]
rubert_prediction = training_adapter.infer([rubert_example["tokens"]])[0]
print(
    f"{rubert_model_type.model_name} · test, документ {rubert_example_index} · "
    f"{len(rubert_example['tokens'].text)} слов · головы без обучения"
)
rubert_visualizer = visualizer_type(
    CLS_LABELS,
    f"RuBERT tiny · fine tuned · test, документ {rubert_example_index}",
    token_evaluator,
    topic_evaluator,
)
rubert_visualizer.visualize(rubert_example, rubert_prediction)

cointegrated/rubert-tiny · test, документ 28 · 184 слов · головы без обучения


Характеристика,Истина,Прогноз,Сравнение
WORKPLACE,Нет,Да,Лишняя
ALTERNATIVE_NAME,Да,Да,Совпадает
WORKS_AS,Да,Да,Совпадает
PARTICIPANT_IN,Да,Да,Совпадает
POINT_IN_TIME,Да,Да,Совпадает
TAKES_PLACE_IN,Да,Да,Совпадает
HEADQUARTERED_IN,Да,Нет,Пропущена
ORIGINS_FROM,Да,Да,Совпадает
LOCATED_IN,Нет,Да,Лишняя
AGE_IS,Да,Да,Совпадает
